In [ ]:

import sys
import tensorflow as tf

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))



# 05 — Model Training

## Objective

Train the CNN using the prepared brain image dataset.

We use:

- Adam optimizer
- Sparse categorical cross-entropy
- Accuracy metric
- Batch size of 32
- Maximum of 15 epochs
- Early stopping to reduce overfitting
- Model checkpointing to save the best validation model

Only training images update model weights.

Validation images are used to monitor generalization.

The test set is not used during training.


In [ ]:

from pathlib import Path
import sys

import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.brainscan import create_dataset, build_model

tf.keras.utils.set_random_seed(42)

manifest_path = PROJECT_ROOT / "data/splits/split_manifest.csv"

manifest = pd.read_csv(manifest_path)

print("Total images:", len(manifest))
print(manifest["split"].value_counts())


def get_records(split_name):
    subset = manifest[manifest["split"] == split_name]

    return [
        (PROJECT_ROOT / path, label)
        for path, label in zip(subset["path"], subset["label"])
    ]


train_records = get_records("train")
val_records = get_records("validation")

train_ds = create_dataset(train_records, training=True)
val_ds = create_dataset(val_records)

gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", gpus)

if not gpus:
    raise RuntimeError("GPU not detected. Check NVIDIA/CUDA setup.")


In [ ]:
model = build_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

best_model_path = MODELS_DIR / "best_cnn.keras"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

print("Best model saved:", best_model_path)


In [ ]:
history_df = pd.DataFrame(history.history)

history_df.to_csv(
    MODELS_DIR / "training_history.csv",
    index=False
)

plt.figure(figsize=(8, 4))

plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")

plt.title("CNN Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))

plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")

plt.title("CNN Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()
